## Notebook 05 — LightGBM Model

### Objective

Train a LightGBM classifier on the same 17 features
and same train/validation split used in notebook 04.

LightGBM is Microsoft's gradient boosting framework.
It uses a different tree building strategy than XGBoost
which means it finds different decision boundaries
on the same data — making it a valuable addition
to the ensemble.

### Why LightGBM Alongside XGBoost

XGBoost builds trees depth-wise — one full level at a time.
LightGBM builds trees leaf-wise — always splitting the
leaf with the highest gain first.

This means:
  XGBoost  →  more balanced trees, conservative splits
  LightGBM →  more aggressive splits, finds deeper patterns

Same data. Different algorithm. Different mistakes.
When both agree on a signal it is much stronger
than either model agreeing with itself.

### What Is Identical to Notebook 04

  Data split   :  2019-2023 train, 2024 validation
  Features     :  same 17 features loaded from feature_cols.pkl
  Rules        :  all 4 critical rules applied identically
  OOF method   :  walk-forward TimeSeriesSplit 5 folds
  Purge gap    :  5 days at every fold boundary
  Assertions   :  same checks after every fold
  Evaluation   :  same metrics on same validation set

### What Is Different

  Model        :  LightGBM instead of XGBoost
  Parameters   :  LightGBM-specific parameter names
  Output files :  lgbm_model.pkl, lgbm_oof_probs.npy

### Notebook Structure

  Cell 1  →  Imports and configuration
  Cell 2  →  Load and verify processed data
  Cell 3  →  Train / Validation split
  Cell 4  →  Define feature columns
  Cell 5  →  Compute class weights
  Cell 6  →  LightGBM parameters
  Cell 7  →  Walk-forward OOF cross-validation
  Cell 8  →  OOF sanity validation
  Cell 9  →  Train final model on full train set
  Cell 10 →  Evaluate on validation set
  Cell 11 →  Feature importance
  Cell 12 →  Save model and OOF predictions

In [1]:
# ============================================================
# CELL 1 - Imports and Configuration
# ============================================================
# Import all libraries needed for this notebook upfront
# Fix all random seeds before any library initialises
# Define all constants in one place for easy modification
# Create output directory for saved models
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection    import TimeSeriesSplit
from sklearn.metrics            import (classification_report,
                                        balanced_accuracy_score,
                                        confusion_matrix,
                                        f1_score)
from sklearn.utils.class_weight import compute_sample_weight

# Fix random seed for full reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Project constants — must match notebook 04 exactly
PURGE_DAYS = 5   # must match prediction horizon in notebook 03
N_SPLITS   = 5   # number of walk-forward CV folds
N_CLASSES  = 3   # Sell=0, Hold=1, Buy=2

# Class label map
LABELS = {0: 'Sell', 1: 'Hold', 2: 'Buy'}

# File paths
PROCESSED_PATH = '../data/processed/all_stocks_features.csv'
MODELS_DIR     = '../models/'
os.makedirs(MODELS_DIR, exist_ok=True)

print('Environment ready')
print('=' * 45)
print(f'  LightGBM version : {lgb.__version__}')
print(f'  Random state     : {RANDOM_STATE}')
print(f'  Purge days       : {PURGE_DAYS}')
print(f'  CV folds         : {N_SPLITS}')
print(f'  Classes          : {LABELS}')
print(f'  Models directory : {MODELS_DIR}')

Environment ready
  LightGBM version : 4.6.0
  Random state     : 42
  Purge days       : 5
  CV folds         : 5
  Classes          : {0: 'Sell', 1: 'Hold', 2: 'Buy'}
  Models directory : ../models/


------------------------------------------

### Cell 2 — Load and Verify Processed Data

#### Why This Is Identical to Notebook 04

Both models train on exactly the same dataset.
Same rows. Same columns. Same order.

This is a fundamental requirement for stacking.
If XGBoost and LightGBM train on different data
their OOF predictions are not comparable.
The meta-model in notebook 07 would be combining
predictions from different populations — meaningless.

#### What We Load Differently From Notebook 04

feature_cols.pkl is loaded from disk here instead
of being defined manually.

This is the list saved by notebook 04.
Loading it guarantees we use the exact same 17 features
in the exact same order as XGBoost.

Column order matters for model predictions.
If LightGBM trains with features in a different order
than XGBoost the stacking meta-model receives
inconsistent inputs and produces wrong combinations.

Loading from disk eliminates any risk of typo
or accidental reordering.

#### Assertions Are Identical to Notebook 04

Same shape, same NaN check, same ticker count.
If anything changed in the processed file between
running notebook 04 and notebook 05 we catch it here.

In [2]:
# ============================================================
# CELL 2 - Load and Verify Processed Data
# ============================================================
# Load same processed file as notebook 04
# Load feature_cols.pkl from disk — do not redefine manually
# This guarantees identical feature order across all models
# Drop macd and macd_signal — same as notebook 04
# Run same hard assertions as notebook 04
# ============================================================

# Load processed data
df = pd.read_csv(PROCESSED_PATH, parse_dates=['Date'])
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

print('Processed data loaded')
print('=' * 50)
print(f'  Shape      : {df.shape}')
print(f'  NaNs       : {df.isnull().sum().sum()}')
print(f'  Tickers    : {df["Ticker"].nunique()}')
print(f'  Date range : {df["Date"].min().date()} '
      f'to {df["Date"].max().date()}')

# Hard assertions
assert df.shape[0]             == 29100, \
    f'Expected 29100 rows, got {df.shape[0]}'
assert df.shape[1]             == 27, \
    f'Expected 27 columns, got {df.shape[1]}'
assert df.isnull().sum().sum() == 0, \
    'NaNs found — check notebook 03 output'
assert df['Ticker'].nunique()  == 20, \
    f'Expected 20 tickers, got {df["Ticker"].nunique()}'

print('\nAll load assertions passed ✅')

# Drop redundant MACD columns — same as notebook 04
df = df.drop(columns=['macd', 'macd_signal'])

assert df.shape[1]     == 25, \
    f'Expected 25 columns after drop, got {df.shape[1]}'
assert 'macd'          not in df.columns, \
    'macd still present'
assert 'macd_signal'   not in df.columns, \
    'macd_signal still present'
assert 'macd_hist'     in df.columns, \
    'macd_hist missing'

print(f'\nDropped        : macd, macd_signal')
print(f'Shape after drop : {df.shape}')
print('Column drop verified ✅')

# Load feature columns from disk
# Do NOT redefine — must match notebook 04 exactly
FEATURE_COLS = joblib.load(f'{MODELS_DIR}feature_cols.pkl')
TARGET_COL   = 'target'

print(f'\nFeature columns loaded from disk')
print(f'  Source   : {MODELS_DIR}feature_cols.pkl')
print(f'  Features : {len(FEATURE_COLS)}')
print(f'  Target   : {TARGET_COL}')

# Verify all features exist in dataframe
missing = [c for c in FEATURE_COLS if c not in df.columns]
assert len(missing) == 0, \
    f'Missing features: {missing}'

# Verify feature count matches notebook 04
assert len(FEATURE_COLS) == 17, \
    f'Expected 17 features, got {len(FEATURE_COLS)}'

print('\nAll feature assertions passed ✅')
print()
print('Features loaded (must match notebook 04 exactly):')
for i, col in enumerate(FEATURE_COLS, 1):
    print(f'  {i:>2}. {col}')

Processed data loaded
  Shape      : (29100, 27)
  NaNs       : 0
  Tickers    : 20
  Date range : 2019-03-14 to 2024-12-20

All load assertions passed ✅

Dropped        : macd, macd_signal
Shape after drop : (29100, 25)
Column drop verified ✅

Feature columns loaded from disk
  Source   : ../models/feature_cols.pkl
  Features : 17
  Target   : target

All feature assertions passed ✅

Features loaded (must match notebook 04 exactly):
   1. return_1d
   2. return_5d
   3. return_10d
   4. return_20d
   5. price_vs_ma20
   6. price_vs_ma50
   7. rsi_14
   8. macd_hist
   9. bb_width
  10. atr_14
  11. volume_ratio
  12. volume_trend
  13. price_volume_signal
  14. market_return_1d
  15. day_of_week
  16. month
  17. quarter


--------------------

### Cell 3 — Train / Validation Split by Date

#### Must Be Identical to Notebook 04

The split boundary must be exactly the same as notebook 04.

  Train      :  2019-03-14  to  2023-12-31
  
  Validation :  2024-01-01  to  2024-12-20

If LightGBM uses a different split boundary:
  OOF predictions cover a different time period
  Validation predictions are on different rows
  Stacking in notebook 07 combines incompatible predictions
  The meta-model produces meaningless output

Same boundary. Same reset_index. Same assertions.
No exceptions.

#### The Vault Rule Still Applies

Validation set locked immediately after this cell.
Not touched again until Cell 10 final evaluation.
No tuning, no peeking, no adjustments based on
validation data between this cell and Cell 10.

In [3]:
# ============================================================
# CELL 3 - Train / Validation Split by Date
# ============================================================
# Identical split to notebook 04
# Same boundary ensures OOF predictions are compatible
# reset_index mandatory — prevents index out of bounds
# in OOF storage loop in Cell 7
# ============================================================

TRAIN_END = '2023-12-31'
VAL_START = '2024-01-01'

df_train = df[df['Date'] <= TRAIN_END].copy()
df_val   = df[df['Date'] >= VAL_START].copy()

# Reset index — critical for OOF array alignment in Cell 7
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)

print('Train / Validation split complete')
print('=' * 55)
print(f'  Train      : {df_train["Date"].min().date()} '
      f'to {df_train["Date"].max().date()} '
      f'| {len(df_train):,} rows '
      f'({len(df_train)/len(df)*100:.1f}%)')
print(f'  Validation : {df_val["Date"].min().date()} '
      f'to {df_val["Date"].max().date()} '
      f'| {len(df_val):,} rows '
      f'({len(df_val)/len(df)*100:.1f}%)')
print(f'  Total      : {len(df_train) + len(df_val):,} rows')
print()
print('  Live 2025  : downloaded from yfinance in dashboard')
print('  Live 2026  : real-time predictions in production')

# Verify no date overlap
assert df_train['Date'].max() < df_val['Date'].min(), \
    'DATE OVERLAP — train and val overlap — LEAKAGE'

# Verify all 20 tickers in both sets
assert df_train['Ticker'].nunique() == 20, \
    f'Train missing tickers'
assert df_val['Ticker'].nunique()   == 20, \
    f'Val missing tickers'

# Verify row counts add up
assert len(df_train) + len(df_val) == len(df), \
    'Row count mismatch'

# Verify index reset correctly
assert df_train.index[0]  == 0, \
    'df_train index does not start at 0'
assert df_train.index[-1] == len(df_train) - 1, \
    'df_train index not contiguous'

# Verify split matches notebook 04 exactly
assert len(df_train) == 24180, \
    f'Train size {len(df_train)} differs from notebook 04 (24180)'
assert len(df_val)   == 4920, \
    f'Val size {len(df_val)} differs from notebook 04 (4920)'

print()
print('Split assertions passed ✅')
print('Split matches notebook 04 exactly ✅')
print()

# Class distribution — should match notebook 04
print('Class distribution comparison:')
print(f'  {"Class":<8} {"Train %":>8} {"Val %":>8} {"Difference":>12}')
print(f'  {"-" * 40}')
for cls in range(N_CLASSES):
    train_pct = (df_train['target'] == cls).mean() * 100
    val_pct   = (df_val['target']   == cls).mean() * 100
    diff      = val_pct - train_pct
    label     = LABELS[cls]
    flag      = '⚠️' if abs(diff) > 5 else '✅'
    print(f'  {label:<8} {train_pct:>8.1f}% {val_pct:>8.1f}% '
          f'{diff:>+11.1f}%  {flag}')

print()
print('Validation set is now LOCKED until Cell 10 ✅')

Train / Validation split complete
  Train      : 2019-03-14 to 2023-12-29 | 24,180 rows (83.1%)
  Validation : 2024-01-02 to 2024-12-20 | 4,920 rows (16.9%)
  Total      : 29,100 rows

  Live 2025  : downloaded from yfinance in dashboard
  Live 2026  : real-time predictions in production

Split assertions passed ✅
Split matches notebook 04 exactly ✅

Class distribution comparison:
  Class     Train %    Val %   Difference
  ----------------------------------------
  Sell         24.9%     20.7%        -4.2%  ✅
  Hold         42.5%     45.9%        +3.5%  ✅
  Buy          32.6%     33.4%        +0.8%  ✅

Validation set is now LOCKED until Cell 10 ✅


------------------

### Cell 4 — Compute Class Weights

#### Identical to Notebook 04

Same training set. Same class distribution.
Same compute_sample_weight call.

LightGBM handles class imbalance differently
from XGBoost internally but we pass sample weights
the same way — through the fit() call.

LightGBM also has a class_weight parameter but
sample_weight gives us more control and keeps
the approach consistent across all three models.

Consistent weighting across XGBoost and LightGBM
ensures both models are solving the same problem
with the same emphasis on rare classes.
If one model used weights and the other did not
their OOF predictions would not be comparable
and stacking would produce inconsistent results.

In [4]:
# ============================================================
# CELL 4 - Compute Class Weights
# ============================================================
# Identical to notebook 04 — same training set
# same class distribution — same weights
# Consistent weighting across models is mandatory
# for comparable OOF predictions in stacking
# ============================================================

y_train_full = df_train['target'].values.astype(int)

sample_weights = compute_sample_weight(
    class_weight='balanced',
    y=y_train_full
)

print('Class weights computed')
print('=' * 55)
print(f'  {"Class":<8} {"Rows":>8} {"Pct":>8} '
      f'{"Weight":>10} {"Meaning"}')
print(f'  {"-" * 50}')

unique_classes, counts = np.unique(
    y_train_full, return_counts=True
)
for cls, count in zip(unique_classes, counts):
    pct    = count / len(y_train_full) * 100
    weight = sample_weights[y_train_full == cls][0]
    label  = LABELS[cls]
    if weight > 1.0:
        meaning = 'more attention'
    elif weight < 1.0:
        meaning = 'less attention'
    else:
        meaning = 'neutral'
    print(f'  {label:<8} {count:>8,} {pct:>8.1f}% '
          f'{weight:>10.4f}  {meaning}')

print()
print(f'  Total rows         : {len(sample_weights):,}')
print(f'  Weight array shape : {sample_weights.shape}')
print(f'  Min weight         : {sample_weights.min():.4f}')
print(f'  Max weight         : {sample_weights.max():.4f}')
print()

# Verify weights are positive
assert (sample_weights > 0).all(), \
    'Negative weights found'

# Verify shape matches training set
assert len(sample_weights) == len(df_train), \
    f'Weight count {len(sample_weights)} != ' \
    f'train rows {len(df_train)}'

# Verify weights match notebook 04
# Same training set must produce identical weights
assert abs(sample_weights.min() - 0.7848) < 0.001, \
    f'Min weight differs from notebook 04 — split mismatch'
assert abs(sample_weights.max() - 1.3378) < 0.001, \
    f'Max weight differs from notebook 04 — split mismatch'

print('All weight assertions passed ✅')
print()
print('Weights match notebook 04 ✅')
print('Note: weights passed to model.fit() only')
print('      never applied to validation or evaluation')

Class weights computed
  Class        Rows      Pct     Weight Meaning
  --------------------------------------------------
  Sell        6,025     24.9%     1.3378  more attention
  Hold       10,270     42.5%     0.7848  less attention
  Buy         7,885     32.6%     1.0222  more attention

  Total rows         : 24,180
  Weight array shape : (24180,)
  Min weight         : 0.7848
  Max weight         : 1.3378

All weight assertions passed ✅

Weights match notebook 04 ✅
Note: weights passed to model.fit() only
      never applied to validation or evaluation


---------------------------

### Cell 5 — LightGBM Parameters

#### Where LightGBM Differs From XGBoost

XGBoost and LightGBM are both gradient boosting
frameworks but they grow trees differently.

XGBoost — depth-wise growth:
  Grows one complete level at a time.
  All leaves at depth 1 split before any leaf at depth 2.
  More balanced trees.
  More conservative — less overfit risk.
  Slower on large datasets.

LightGBM — leaf-wise growth:
  Always splits the single leaf with highest gain.
  Grows unbalanced trees that go deep on informative paths.
  Finds better splits faster.
  Can overfit more easily — controlled by num_leaves.
  Much faster on large datasets.

This means LightGBM will find different patterns
than XGBoost on the same data. Some stocks will get
better Buy signals from LightGBM, others from XGBoost.
The ensemble captures both sets of patterns.

#### Key Parameters Explained

objective = multiclass
  LightGBM equivalent of XGBoost multi:softprob.
  Tells LightGBM this is a multi-class problem.

num_class = 3
  Three output classes — Sell=0, Hold=1, Buy=2.

n_estimators = 500
  Same as XGBoost for fair comparison.
  500 boosting rounds.

learning_rate = 0.05
  Same as XGBoost — controls step size per tree.
  Lower = more trees needed = less overfit.

num_leaves = 31
  Maximum number of leaves per tree.
  This is the primary complexity control in LightGBM.
  XGBoost uses max_depth — LightGBM uses num_leaves.
  31 is the LightGBM default — good baseline.
  A tree with max_depth=6 has at most 2^6=64 leaves.
  31 leaves gives slightly simpler trees than XGBoost.

min_child_samples = 20
  Minimum rows required to form a leaf.
  Prevents the model from creating leaves that represent
  only a handful of rows — reduces overfitting.
  Important for LightGBM because leaf-wise growth
  can create very specific leaves without this constraint.

subsample = 0.8
  Same as XGBoost — each tree sees 80% of rows.

colsample_bytree = 0.8
  Same as XGBoost — each tree sees 80% of features.
  Called feature_fraction in native LightGBM API
  but colsample_bytree works in sklearn wrapper.

metric = multi_logloss
  LightGBM equivalent of XGBoost mlogloss.
  Multiclass log loss for monitoring training.

verbosity = -1
  Suppress all LightGBM training output.
  LightGBM is very verbose by default.
  We control what gets printed ourselves.

In [5]:
# ============================================================
# CELL 5 - LightGBM Parameters
# ============================================================
# Define all parameters in one dictionary
# Same dict used in Cell 6 (OOF loop) and Cell 8 (final)
# Key difference from XGBoost: num_leaves instead of max_depth
# verbosity=-1 suppresses LightGBM default verbose output
# ============================================================

LGBM_PARAMS = {
    # Task definition
    'objective'        : 'multiclass',
    'num_class'        : N_CLASSES,
    'metric'           : 'multi_logloss',

    # Tree structure
    'n_estimators'     : 500,
    'learning_rate'    : 0.05,
    'num_leaves'       : 31,
    'min_child_samples': 20,

    # Randomisation — prevents overfitting
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,

    # Reproducibility and output
    'random_state'     : RANDOM_STATE,
    'n_jobs'           : -1,
    'verbosity'        : -1
}

print('LightGBM baseline parameters defined')
print('=' * 50)
print(f'  {"Parameter":<22} {"Value"}')
print(f'  {"-" * 40}')
for param, value in LGBM_PARAMS.items():
    print(f'  {param:<22} {value}')

print()
print('Key differences from XGBoost:')
print('  max_depth   →  not used in LightGBM')
print('  num_leaves  →  31  primary complexity control')
print('  min_child_samples → 20  prevents tiny leaves')
print('  verbosity   →  -1  suppress LightGBM output')
print()
print('These parameters are used in:')
print('  Cell 6  →  OOF cross-validation loop')
print('  Cell 8  →  Final model on full train set')
print()

# Sanity checks
assert LGBM_PARAMS['num_class']    == N_CLASSES, \
    'num_class must match N_CLASSES'
assert LGBM_PARAMS['objective']    == 'multiclass', \
    'objective must be multiclass'
assert LGBM_PARAMS['random_state'] == RANDOM_STATE, \
    'random_state must match RANDOM_STATE'

print('Parameter assertions passed ✅')

LightGBM baseline parameters defined
  Parameter              Value
  ----------------------------------------
  objective              multiclass
  num_class              3
  metric                 multi_logloss
  n_estimators           500
  learning_rate          0.05
  num_leaves             31
  min_child_samples      20
  subsample              0.8
  colsample_bytree       0.8
  random_state           42
  n_jobs                 -1
  verbosity              -1

Key differences from XGBoost:
  max_depth   →  not used in LightGBM
  num_leaves  →  31  primary complexity control
  min_child_samples → 20  prevents tiny leaves
  verbosity   →  -1  suppress LightGBM output

These parameters are used in:
  Cell 6  →  OOF cross-validation loop
  Cell 8  →  Final model on full train set

Parameter assertions passed ✅


-----------

### Cell 6 — Walk-Forward Out-of-Fold Cross Validation

#### Identical Structure to Notebook 04

All four critical rules applied identically:

Rule 1 — Purge Gap
  Last 5 dates dropped from every training fold.
  Same PURGE_DAYS = 5 matching prediction horizon.

Rule 2 — Date-Based Splitting
  Split on unique dates not row indices.
  All 20 tickers included in every fold correctly.

Rule 3 — Assert After Every Fold
  Date overlap check before fitting.
  Ticker count check before fitting.
  Fail loud — never proceed on corrupted fold.

Rule 4 — OOF Coverage Tracking
  oof_coverage tracks predictions per row.
  Must be 0 or 1 for every row after loop.
  Checked in Cell 7 sanity validation.

#### What Is Different From Notebook 04

Only the model object changes:
  Notebook 04 → xgb.XGBClassifier(**XGB_PARAMS)
  Notebook 05 → lgb.LGBMClassifier(**LGBM_PARAMS)

Everything else — split logic, purge logic,
assert logic, storage logic — is identical.

This is intentional. Consistency in the training
pipeline is what makes the OOF predictions
comparable and stackable in notebook 07.

#### Expected Fold Scores

LightGBM often performs similarly to XGBoost
on tabular financial data at baseline.
Expect F1 macro in the range 0.35 to 0.42.

LightGBM may show more variance across folds
because leaf-wise growth is more sensitive
to the specific data distribution in each fold.
A higher std than XGBoost is normal and expected.

In [6]:
# ============================================================
# CELL 6 - Walk-Forward OOF Cross-Validation
# ============================================================
# Identical structure to notebook 04 Cell 7
# Only the model object changes — LGBMClassifier
# Rule 1 : purge last 5 dates from each training fold
# Rule 2 : split on unique dates not row indices
# Rule 3 : assert checks after every fold split
# Rule 4 : track OOF coverage — catch gaps and duplicates
# ============================================================

# Get sorted unique dates from training set only
unique_dates = np.array(sorted(df_train['Date'].unique()))

tscv = TimeSeriesSplit(n_splits=N_SPLITS)

# Initialise OOF arrays
oof_probs    = np.full((len(df_train), N_CLASSES), np.nan)
oof_coverage = np.zeros(len(df_train), dtype=int)

fold_scores = []

print('Walk-Forward OOF Cross-Validation')
print('=' * 65)
print(f'  Total unique dates in train : {len(unique_dates)}')
print(f'  Number of folds             : {N_SPLITS}')
print(f'  Purge days per boundary     : {PURGE_DAYS}')
print('=' * 65)

for fold, (train_idx, val_idx) in enumerate(
        tscv.split(unique_dates)):

    # ── Rule 2 : date-based splitting ─────────────────────
    train_dates = unique_dates[train_idx]
    val_dates   = unique_dates[val_idx]

    # ── Rule 1 : purge last 5 dates from training fold ────
    train_dates_purged = train_dates[:-PURGE_DAYS]

    # Filter dataframe by purged dates
    fold_train = df_train[
        df_train['Date'].isin(train_dates_purged)
    ]
    fold_val = df_train[
        df_train['Date'].isin(val_dates)
    ]

    # ── Rule 3 : assert checks ────────────────────────────
    assert fold_train['Date'].max() < fold_val['Date'].min(), \
        f'Fold {fold+1}: date overlap — LEAKAGE DETECTED'
    assert fold_train['Ticker'].nunique() == 20, \
        f'Fold {fold+1}: only ' \
        f'{fold_train["Ticker"].nunique()} tickers in train'
    assert fold_val['Ticker'].nunique() == 20, \
        f'Fold {fold+1}: only ' \
        f'{fold_val["Ticker"].nunique()} tickers in val'

    # ── Prepare arrays ────────────────────────────────────
    X_fold_train = fold_train[FEATURE_COLS].values
    y_fold_train = fold_train['target'].values.astype(int)
    X_fold_val   = fold_val[FEATURE_COLS].values
    y_fold_val   = fold_val['target'].values.astype(int)

    # Compute sample weights for this fold only
    fold_weights = compute_sample_weight(
        'balanced', y=y_fold_train
    )

    # ── Train LightGBM on this fold ───────────────────────
    model_fold = lgb.LGBMClassifier(**LGBM_PARAMS)
    model_fold.fit(
        X_fold_train, y_fold_train,
        sample_weight=fold_weights
    )

    # ── Generate OOF predictions ──────────────────────────
    fold_probs = model_fold.predict_proba(X_fold_val)

    # ── Store aligned to df_train positional index ────────
    # df_train was reset_index in Cell 3
    # index runs cleanly from 0 to 24179
    val_positions = df_train[
        df_train['Date'].isin(val_dates)
    ].index.tolist()

    oof_probs[val_positions]     = fold_probs
    oof_coverage[val_positions] += 1

    # ── Fold metrics ──────────────────────────────────────
    fold_preds = np.argmax(fold_probs, axis=1)
    fold_f1    = f1_score(
        y_fold_val, fold_preds, average='macro'
    )
    fold_bal   = balanced_accuracy_score(
        y_fold_val, fold_preds
    )
    fold_scores.append(fold_f1)

    print(f'\nFold {fold+1} / {N_SPLITS}')
    print(f'  Train : {fold_train["Date"].min().date()} '
          f'to {fold_train["Date"].max().date()} '
          f'| {len(fold_train):,} rows')
    print(f'  Val   : {fold_val["Date"].min().date()} '
          f'to {fold_val["Date"].max().date()} '
          f'| {len(fold_val):,} rows')
    print(f'  Purged: last {PURGE_DAYS} dates removed '
          f'from training boundary')
    print(f'  F1 Macro          : {fold_f1:.4f}')
    print(f'  Balanced Accuracy : {fold_bal:.4f}')

print()
print('=' * 65)
print('OOF Cross-Validation Complete')
print(f'  Mean F1 (macro)   : {np.mean(fold_scores):.4f}')
print(f'  Std  F1           : {np.std(fold_scores):.4f}')
print(f'  Min  F1           : {np.min(fold_scores):.4f}')
print(f'  Max  F1           : {np.max(fold_scores):.4f}')
print()
print('OOF arrays ready for Cell 7 sanity validation')
print(f'  oof_probs shape    : {oof_probs.shape}')
print(f'  oof_coverage shape : {oof_coverage.shape}')

Walk-Forward OOF Cross-Validation
  Total unique dates in train : 1209
  Number of folds             : 5
  Purge days per boundary     : 5

Fold 1 / 5
  Train : 2019-03-14 to 2019-12-24 | 3,980 rows
  Val   : 2020-01-03 to 2020-10-19 | 4,020 rows
  Purged: last 5 dates removed from training boundary
  F1 Macro          : 0.3606
  Balanced Accuracy : 0.3619

Fold 2 / 5
  Train : 2019-03-14 to 2020-10-12 | 8,000 rows
  Val   : 2020-10-20 to 2021-08-06 | 4,020 rows
  Purged: last 5 dates removed from training boundary
  F1 Macro          : 0.3648
  Balanced Accuracy : 0.3667

Fold 3 / 5
  Train : 2019-03-14 to 2021-07-30 | 12,020 rows
  Val   : 2021-08-09 to 2022-05-24 | 4,020 rows
  Purged: last 5 dates removed from training boundary
  F1 Macro          : 0.3600
  Balanced Accuracy : 0.3697

Fold 4 / 5
  Train : 2019-03-14 to 2022-05-17 | 16,040 rows
  Val   : 2022-05-25 to 2023-03-14 | 4,020 rows
  Purged: last 5 dates removed from training boundary
  F1 Macro          : 0.3598
  Balanc

-------------

### Cell 7 — OOF Sanity Validation

#### Identical to Notebook 04 Cell 8

Same six checks. Same assertions. Same philosophy.

The checks do not change between notebooks because
the failure modes do not change between notebooks.
Silent corruption in LightGBM OOF predictions is
just as dangerous as in XGBoost OOF predictions.

Both OOF arrays feed into the same meta-model
in notebook 07. A corrupted LightGBM OOF array
poisons the stacking layer just as badly as
a corrupted XGBoost array would.

Never skip this cell. Never comment out assertions.

#### The Six Checks

Check 1 — No row predicted twice
  oof_coverage must never exceed 1.
  Overlap in fold validation sets = corrupted predictions.

Check 2 — Coverage count is sensible
  Predicted + unpredicted = total training rows.
  Unpredicted rows are first fold training rows only.

Check 3 — Probabilities sum to 1.0
  P(Sell) + P(Hold) + P(Buy) = 1.0 for every row.
  LightGBM softmax output must be valid probabilities.

Check 4 — No NaN in predicted rows
  Every predicted row must have real numbers.
  NaN means a fold failed to write predictions.

Check 5 — Shape matches training set
  oof_probs must be (24180, 3) exactly.
  Any other shape means array was corrupted.

Check 6 — Class distribution is reasonable
  OOF predicted distribution vs true distribution.
  Deviation under 15% per class is acceptable.
  Larger deviation is a warning — investigate before
  proceeding to notebook 07.

In [7]:
# ============================================================
# CELL 7 - OOF Sanity Validation (6 Checks)
# ============================================================
# Identical to notebook 04 Cell 8
# All 6 checks must pass before proceeding to Cell 8
# A single failure stops execution with a clear message
# This protects notebook 07 from corrupted stacking input
# Never skip or comment out these checks
# ============================================================

print('OOF Sanity Validation — 6 Checks')
print('=' * 55)

# ── Check 1 — No row predicted twice ──────────────────────
double_predicted = (oof_coverage > 1).sum()
assert double_predicted == 0, \
    f'CHECK 1 FAILED: {double_predicted} rows predicted ' \
    f'more than once — date overlap in fold splitting'
print(f'  Check 1 — No double predictions     : PASSED ✅'
      f'  (max coverage = {oof_coverage.max()})')

# ── Check 2 — Coverage count is sensible ──────────────────
zero_coverage   = (oof_coverage == 0).sum()
predicted_count = (oof_coverage == 1).sum()

print(f'  Check 2 — Coverage summary:')
print(f'            Predicted rows   : {predicted_count:,}')
print(f'            Unpredicted rows : {zero_coverage:,} '
      f'(first fold train rows — expected)')
print(f'            Total rows       : {len(df_train):,}')

assert predicted_count + zero_coverage == len(df_train), \
    f'CHECK 2 FAILED: predicted {predicted_count} + ' \
    f'unpredicted {zero_coverage} != {len(df_train)}'
print(f'  Check 2 — Coverage count sensible   : PASSED ✅')

# ── Check 3 — Probabilities sum to 1.0 ────────────────────
predicted_mask = oof_coverage == 1
prob_sums      = oof_probs[predicted_mask].sum(axis=1)
max_deviation  = np.abs(prob_sums - 1.0).max()
assert max_deviation < 1e-5, \
    f'CHECK 3 FAILED: probabilities do not sum to 1.0 ' \
    f'max deviation = {max_deviation:.2e}'
print(f'  Check 3 — Probs sum to 1.0          : PASSED ✅'
      f'  (max deviation = {max_deviation:.2e})')

# ── Check 4 — No NaN in predicted rows ────────────────────
nan_in_predicted = np.isnan(
    oof_probs[predicted_mask]
).sum()
assert nan_in_predicted == 0, \
    f'CHECK 4 FAILED: {nan_in_predicted} NaN values ' \
    f'in predicted rows — fold storage failed'
print(f'  Check 4 — No NaN in predictions     : PASSED ✅')

# ── Check 5 — Shape matches training set ──────────────────
expected_shape = (len(df_train), N_CLASSES)
assert oof_probs.shape == expected_shape, \
    f'CHECK 5 FAILED: expected {expected_shape} ' \
    f'got {oof_probs.shape}'
print(f'  Check 5 — OOF shape correct         : PASSED ✅'
      f'  {oof_probs.shape}')

# ── Check 6 — Class distribution comparison ───────────────
oof_preds      = np.argmax(oof_probs[predicted_mask], axis=1)
y_true_covered = df_train['target'].values[predicted_mask]

print(f'  Check 6 — Class distribution comparison:')
print(f'            {"Class":<8} {"True %":>8} '
      f'{"OOF Pred %":>12} {"Diff":>8} {"Status"}')
print(f'            {"-" * 50}')

all_cls_ok = True
for cls in range(N_CLASSES):
    true_pct = (y_true_covered == cls).mean() * 100
    pred_pct = (oof_preds      == cls).mean() * 100
    diff     = abs(pred_pct - true_pct)
    label    = LABELS[cls]
    flag     = '✅' if diff < 15 else '⚠️  large deviation'
    if diff >= 15:
        all_cls_ok = False
    print(f'            {label:<8} {true_pct:>8.1f}% '
          f'{pred_pct:>11.1f}% {diff:>7.1f}%  {flag}')

if all_cls_ok:
    print(f'  Check 6 — Distribution reasonable   : PASSED ✅')
else:
    print(f'  Check 6 — Distribution WARNING      : ⚠️')
    print(f'            Investigate before proceeding')

# ── Cross-notebook consistency check ──────────────────────
# Predicted count must match notebook 04
# Same fold structure = same coverage pattern
xgb_coverage = np.load(
    f'{MODELS_DIR}xgb_oof_coverage.npy'
)
xgb_predicted = (xgb_coverage == 1).sum()
assert predicted_count == xgb_predicted, \
    f'CONSISTENCY FAILED: LightGBM predicted {predicted_count} ' \
    f'rows but XGBoost predicted {xgb_predicted} rows ' \
    f'— fold structure mismatch'
print()
print(f'  Cross-notebook consistency check:')
print(f'  XGBoost  predicted rows : {xgb_predicted:,}')
print(f'  LightGBM predicted rows : {predicted_count:,}')
print(f'  Match                   : ✅')

# ── Final summary ──────────────────────────────────────────
print()
print('=' * 55)
print('All OOF sanity checks passed ✅')
print('OOF predictions are clean and ready')
print()
print('Next steps:')
print('  Cell 8       →  train final model on full train set')
print('  Notebook 07  →  load oof_probs to train meta-model')
print(f'  Predicted rows available : {predicted_count:,}')
print(f'  OOF array shape          : {oof_probs.shape}')

OOF Sanity Validation — 6 Checks
  Check 1 — No double predictions     : PASSED ✅  (max coverage = 1)
  Check 2 — Coverage summary:
            Predicted rows   : 20,100
            Unpredicted rows : 4,080 (first fold train rows — expected)
            Total rows       : 24,180
  Check 2 — Coverage count sensible   : PASSED ✅
  Check 3 — Probs sum to 1.0          : PASSED ✅  (max deviation = 2.22e-16)
  Check 4 — No NaN in predictions     : PASSED ✅
  Check 5 — OOF shape correct         : PASSED ✅  (24180, 3)
  Check 6 — Class distribution comparison:
            Class      True %   OOF Pred %     Diff Status
            --------------------------------------------------
            Sell         26.4%        22.0%     4.4%  ✅
            Hold         40.3%        41.3%     1.0%  ✅
            Buy          33.3%        36.8%     3.5%  ✅
  Check 6 — Distribution reasonable   : PASSED ✅

  Cross-notebook consistency check:
  XGBoost  predicted rows : 20,100
  LightGBM predicted rows : 20

---------------

### Cell 8 — Train Final LightGBM on Full Training Set

#### Identical Logic to Notebook 04 Cell 9

Train one final model on all 24,180 training rows.
Same reasoning as XGBoost:

  OOF loop trained 5 models on partial data.
  Those models generated honest OOF predictions.
  Those models are now discarded.

  This final model trains on everything.
  More data = better learned patterns.
  This model goes to disk for production use.

#### What to Expect

LightGBM trains significantly faster than XGBoost
because leaf-wise growth finds good splits faster.
Expect training to complete in under 30 seconds.

Training set F1 will again be high — around 0.80+.
This is expected memorisation not leakage.
The honest score is the validation score in Cell 9.

The gap between train F1 and OOF F1 may be
slightly larger for LightGBM than XGBoost
because leaf-wise growth memorises training data
more aggressively than depth-wise growth.
This is normal LightGBM behaviour.

In [8]:
# ============================================================
# CELL 8 - Train Final LightGBM on Full Training Set
# ============================================================
# Train ONE model on all 24,180 training rows
# Same parameters as CV loop — consistency is mandatory
# This model is saved to disk and used in production
# The 5 fold models from Cell 6 are discarded
# ============================================================

X_train_full = df_train[FEATURE_COLS].values
y_train_full = df_train['target'].values.astype(int)
w_train_full = compute_sample_weight(
    'balanced', y=y_train_full
)

print('Training final LightGBM on full training set')
print('=' * 50)
print(f'  Training rows : {len(X_train_full):,}')
print(f'  Features      : {len(FEATURE_COLS)}')
print(f'  Classes       : {N_CLASSES}')
print(f'  Parameters    : same as CV loop')
print()
print('Training in progress...')

lgbm_final = lgb.LGBMClassifier(**LGBM_PARAMS)
lgbm_final.fit(
    X_train_full,
    y_train_full,
    sample_weight=w_train_full
)

print('Training complete')
print()

# Quick sanity check on trained model
train_preds  = lgbm_final.predict(X_train_full)
train_probs  = lgbm_final.predict_proba(X_train_full)

train_f1  = f1_score(
    y_train_full, train_preds, average='macro'
)
train_bal = balanced_accuracy_score(
    y_train_full, train_preds
)

print('Final model — Training Set Performance')
print('=' * 50)
print(f'  F1 Macro          : {train_f1:.4f}')
print(f'  Balanced Accuracy : {train_bal:.4f}')
print()
print('  Note: training set scores are always higher')
print('  than OOF scores — model has seen these rows.')
print(f'  OOF Mean F1 was   : {np.mean(fold_scores):.4f}')
print(f'  Train F1 is       : {train_f1:.4f}')

gap = train_f1 - np.mean(fold_scores)
print(f'  Gap               : {gap:.4f}')
if gap > 0.15:
    print('  ⚠️  Gap > 0.15 — LightGBM memorising training data')
    print('     This is normal for leaf-wise growth')
    print('     Validation score in Cell 9 is what matters')
else:
    print('  ✅ Gap is reasonable')

print()
print(f'  Probability output shape : {train_probs.shape}')
assert train_probs.shape == (len(X_train_full), N_CLASSES), \
    'Probability shape mismatch'

# Compare train F1 to XGBoost train F1
# LightGBM may memorise more aggressively
print()
print('  Note: LightGBM leaf-wise growth often produces')
print('  higher train F1 than XGBoost depth-wise growth.')
print('  Gap larger than notebook 04 is expected.')
print()
print('Final model ready for Cell 9 evaluation ✅')

Training final LightGBM on full training set
  Training rows : 24,180
  Features      : 17
  Classes       : 3
  Parameters    : same as CV loop

Training in progress...
Training complete

Final model — Training Set Performance
  F1 Macro          : 0.8409
  Balanced Accuracy : 0.8430

  Note: training set scores are always higher
  than OOF scores — model has seen these rows.
  OOF Mean F1 was   : 0.3687
  Train F1 is       : 0.8409
  Gap               : 0.4721
  ⚠️  Gap > 0.15 — LightGBM memorising training data
     This is normal for leaf-wise growth
     Validation score in Cell 9 is what matters

  Probability output shape : (24180, 3)

  Note: LightGBM leaf-wise growth often produces
  higher train F1 than XGBoost depth-wise growth.
  Gap larger than notebook 04 is expected.

Final model ready for Cell 9 evaluation ✅


-----------

### Cell 9 — Evaluate on Validation Set

#### The Moment of Truth for LightGBM

Validation set opened here for the first and only time
in this notebook.

We ask one question:
  How well does LightGBM predict stock signals
  on 2024 data it has never seen before?

#### What We Compare

Three comparisons matter here:

1. LightGBM val F1 vs LightGBM OOF F1
   Should be close — confirms generalisation.
   Large gap means overfitting on training data.

2. LightGBM val F1 vs XGBoost val F1
   Should be similar — both are baseline models.
   One clearly beating the other is worth noting.
   The ensemble will outperform both regardless.

3. LightGBM per-class performance vs XGBoost
   More interesting than overall F1.
   If LightGBM is better at Sell and XGBoost
   is better at Buy the ensemble combines both
   strengths automatically through stacking.

#### What Score to Expect

LightGBM OOF mean F1 : 0.3652
Expected val F1       : 0.35 to 0.42
If val F1 close to OOF F1 → generalises well
If val F1 much lower      → overfitting

#### Do Not Retune After Seeing This Score

If val F1 is lower than expected resist the urge
to change parameters and re-evaluate on validation.
That is peeking — validation score becomes optimistic.
All tuning happens in notebook 07 using OOF scores only.

In [9]:
# ============================================================
# CELL 9 - Validation Set Evaluation
# ============================================================
# Validation set opened here for the first and only time
# Compute all classification metrics on 2024 held-out data
# Compare to XGBoost results from notebook 04
# Do NOT retune after seeing these results
# ============================================================

X_val = df_val[FEATURE_COLS].values
y_val = df_val['target'].values.astype(int)

# Generate predictions on validation set
val_probs = lgbm_final.predict_proba(X_val)
val_preds = np.argmax(val_probs, axis=1)

# ── Core metrics ──────────────────────────────────────────
bal_acc = balanced_accuracy_score(y_val, val_preds)
f1_mac  = f1_score(y_val, val_preds, average='macro')
f1_wtd  = f1_score(y_val, val_preds, average='weighted')

print('LightGBM — Validation Set Results (2024)')
print('=' * 55)
print(f'  Balanced Accuracy : {bal_acc:.4f}')
print(f'  F1 Macro          : {f1_mac:.4f}')
print(f'  F1 Weighted       : {f1_wtd:.4f}')
print()

# ── Generalisation check ──────────────────────────────────
oof_mean = np.mean(fold_scores)
val_gap  = abs(f1_mac - oof_mean)
print('  Generalisation Check:')
print(f'  OOF Mean F1  : {oof_mean:.4f}')
print(f'  Val F1       : {f1_mac:.4f}')
print(f'  Gap          : {val_gap:.4f}')
if val_gap < 0.05:
    print('  ✅ Excellent — val score matches OOF closely')
elif val_gap < 0.10:
    print('  ✅ Good — val score within acceptable range')
else:
    print('  ⚠️  Val score differs from OOF significantly')
    print('     Consider tuning in notebook 07')
print()

# ── XGBoost comparison ────────────────────────────────────
# Load XGBoost validation predictions for comparison
xgb_model  = joblib.load(f'{MODELS_DIR}xgb_model.pkl')
xgb_probs  = xgb_model.predict_proba(X_val)
xgb_preds  = np.argmax(xgb_probs, axis=1)
xgb_f1_mac = f1_score(y_val, xgb_preds, average='macro')
xgb_bal    = balanced_accuracy_score(y_val, xgb_preds)

print('  Model Comparison on Validation Set:')
print(f'  {"Model":<12} {"F1 Macro":>10} '
      f'{"Bal Acc":>10} {"Winner"}')
print(f'  {"-" * 45}')
lgbm_f1_win = "✅" if f1_mac  > xgb_f1_mac else "  "
xgb_f1_win  = "✅" if xgb_f1_mac > f1_mac  else "  "
lgbm_bal_win= "✅" if bal_acc > xgb_bal    else "  "
xgb_bal_win = "✅" if xgb_bal > bal_acc    else "  "
print(f'  {"XGBoost":<12} {xgb_f1_mac:>10.4f} '
      f'{xgb_bal:>10.4f} {xgb_f1_win}')
print(f'  {"LightGBM":<12} {f1_mac:>10.4f} '
      f'{bal_acc:>10.4f} {lgbm_f1_win}')
print()

# ── Classification report ─────────────────────────────────
print('Classification Report:')
print('-' * 55)
print(classification_report(
    y_val, val_preds,
    target_names=['Sell', 'Hold', 'Buy'],
    digits=4
))

# ── Confusion matrix ──────────────────────────────────────
print('Confusion Matrix (rows=Actual, cols=Predicted):')
print('-' * 55)
cm = confusion_matrix(y_val, val_preds)
print(f'  {"":>10} {"Pred Sell":>10} '
      f'{"Pred Hold":>10} {"Pred Buy":>10}')
print(f'  {"-" * 42}')
for i, row in enumerate(cm):
    label = f'Act {LABELS[i]}'
    print(f'  {label:>10} {row[0]:>10} '
          f'{row[1]:>10} {row[2]:>10}')
print()

# ── Per class accuracy ────────────────────────────────────
print('Per Class Accuracy — LightGBM vs XGBoost:')
print('-' * 55)
print(f'  {"Class":<6} {"LGBM Acc":>10} '
      f'{"XGB Acc":>10} {"Winner"}')
print(f'  {"-" * 40}')
for cls in range(N_CLASSES):
    cls_mask      = y_val == cls
    lgbm_correct  = (val_preds[cls_mask]  == cls).sum()
    xgb_correct   = (xgb_preds[cls_mask]  == cls).sum()
    cls_total     = cls_mask.sum()
    lgbm_acc      = lgbm_correct / cls_total * 100
    xgb_acc       = xgb_correct  / cls_total * 100
    winner        = 'LGBM ✅' if lgbm_acc > xgb_acc \
                    else 'XGB ✅'
    print(f'  {LABELS[cls]:<6} {lgbm_acc:>9.1f}% '
          f'{xgb_acc:>9.1f}%  {winner}')

print()
print('=' * 55)
print('Validation evaluation complete')
print('Validation set is now closed')
print('Next : Cell 10 — Feature Importance')

LightGBM — Validation Set Results (2024)
  Balanced Accuracy : 0.3742
  F1 Macro          : 0.3711
  F1 Weighted       : 0.4146

  Generalisation Check:
  OOF Mean F1  : 0.3687
  Val F1       : 0.3711
  Gap          : 0.0024
  ✅ Excellent — val score matches OOF closely

  Model Comparison on Validation Set:
  Model          F1 Macro    Bal Acc Winner
  ---------------------------------------------
  XGBoost          0.3765     0.3799 ✅
  LightGBM         0.3711     0.3742   

Classification Report:
-------------------------------------------------------
              precision    recall  f1-score   support

        Sell     0.2145    0.2181    0.2163      1018
        Hold     0.5162    0.6142    0.5609      2260
         Buy     0.3988    0.2905    0.3362      1642

    accuracy                         0.4242      4920
   macro avg     0.3765    0.3742    0.3711      4920
weighted avg     0.4146    0.4242    0.4146      4920

Confusion Matrix (rows=Actual, cols=Predicted):
----------

----------

### Cell 10 — Feature Importance

#### Why We Check Feature Importance Again

We already saw XGBoost feature importance in notebook 04.
Checking LightGBM importance serves three purposes:

1. Validation — confirms LightGBM learned from
   meaningful features not noise or artifacts.

2. Comparison — LightGBM and XGBoost may disagree
   on which features matter most. A feature that
   ranks high in both models is almost certainly
   a genuine signal. A feature that ranks high in
   one but low in the other is model-specific —
   worth noting for notebook 07 tuning.

3. Ensemble insight — if LightGBM relies heavily
   on different features than XGBoost the two
   models are truly complementary. The meta-model
   in notebook 07 benefits from this diversity.

#### LightGBM Importance Types

LightGBM supports the same importance types as XGBoost:

split — number of times feature used to split
        equivalent to XGBoost weight
        can favour features used in many shallow splits

gain  — total gain from all splits using this feature
        more meaningful — reflects actual improvement
        in prediction quality from using each feature

We use gain as the primary ranking — same as notebook 04.
This allows direct comparison between the two models.

#### What Makes This Comparison Valuable

If both models agree:
  atr_14 is important in both → genuine volatility signal
  quarter is important in both → genuine seasonal effect

If models disagree:
  Feature A ranks 2 in XGBoost but 14 in LightGBM
  → XGBoost found a depth-wise pattern LightGBM missed
  → or LightGBM found it redundant given other features
  → worth investigating in notebook 07

The cross-model comparison at the end of this cell
is the most valuable output — it identifies which
features are universally important vs model-specific.

In [10]:
# ============================================================
# CELL 10 - Feature Importance
# ============================================================
# Compute split and gain importance for LightGBM
# Compare directly to XGBoost importance from notebook 04
# Cross-model agreement identifies strongest signals
# Cross-model disagreement identifies model-specific patterns
# ============================================================

# ── LightGBM importance ───────────────────────────────────
lgbm_importance_split = lgbm_final.booster_.feature_importance(
    importance_type='split'
)
lgbm_importance_gain  = lgbm_final.booster_.feature_importance(
    importance_type='gain'
)

# Build dataframe
feat_imp = pd.DataFrame({
    'Feature'    : FEATURE_COLS,
    'Split'      : lgbm_importance_split,
    'Gain'       : lgbm_importance_gain.astype(float)
})

feat_imp['Split_pct'] = (
    feat_imp['Split'] / feat_imp['Split'].sum() * 100
)
feat_imp['Gain_pct'] = (
    feat_imp['Gain'] / feat_imp['Gain'].sum() * 100
)

feat_imp_gain   = feat_imp.sort_values(
    'Gain_pct', ascending=False
).reset_index(drop=True)
feat_imp_split  = feat_imp.sort_values(
    'Split_pct', ascending=False
).reset_index(drop=True)

# ── Print split ranking ───────────────────────────────────
print('LightGBM Feature Importance — Split Ranking')
print('(how often each feature is used to split)')
print('=' * 60)
print(f'  {"Rank":<5} {"Feature":<25} {"Split %":>9} {"Bar"}')
print(f'  {"-" * 55}')
for i, row in feat_imp_split.iterrows():
    bar = '█' * int(row['Split_pct'])
    print(f'  {i+1:<5} {row["Feature"]:<25} '
          f'{row["Split_pct"]:>8.2f}%  {bar}')

print()

# ── Print gain ranking ────────────────────────────────────
print('LightGBM Feature Importance — Gain Ranking')
print('(how much each feature actually improves predictions)')
print('=' * 60)
print(f'  {"Rank":<5} {"Feature":<25} {"Gain %":>9} {"Bar"}')
print(f'  {"-" * 55}')
for i, row in feat_imp_gain.iterrows():
    bar = '█' * int(row['Gain_pct'])
    print(f'  {i+1:<5} {row["Feature"]:<25} '
          f'{row["Gain_pct"]:>8.2f}%  {bar}')

print()

# ── Cross-model comparison ────────────────────────────────
# Load XGBoost importance for comparison
# Recompute from saved model
xgb_model = joblib.load(f'{MODELS_DIR}xgb_model.pkl')
xgb_imp   = xgb_model.get_booster().get_score(
    importance_type='gain'
)
xgb_gain_vals = np.array([
    xgb_imp.get(f'f{i}', 0)
    for i in range(len(FEATURE_COLS))
])
xgb_gain_pct = xgb_gain_vals / xgb_gain_vals.sum() * 100

# Build rank dictionaries
lgbm_gain_rank = {
    row['Feature']: i+1
    for i, row in feat_imp_gain.iterrows()
}
xgb_gain_series = pd.Series(
    xgb_gain_pct, index=FEATURE_COLS
).sort_values(ascending=False)
xgb_gain_rank = {
    feat: i+1
    for i, feat in enumerate(xgb_gain_series.index)
}

print('Cross-Model Feature Importance Comparison')
print('(gain ranking — lower rank = more important)')
print('=' * 65)
print(f'  {"Feature":<25} {"LGBM Rank":>10} '
      f'{"XGB Rank":>10} {"Diff":>6} {"Agreement"}')
print(f'  {"-" * 60}')

agreements   = []
for feat in FEATURE_COLS:
    lr   = lgbm_gain_rank[feat]
    xr   = xgb_gain_rank[feat]
    diff = abs(lr - xr)
    if diff <= 3:
        agreement = '✅ strong'
    elif diff <= 6:
        agreement = '🔶 moderate'
    else:
        agreement = '⚠️  disagree'
    agreements.append({
        'Feature'  : feat,
        'LGBM_rank': lr,
        'XGB_rank' : xr,
        'Diff'     : diff,
        'Agreement': agreement
    })
    print(f'  {feat:<25} {lr:>10} {xr:>10} '
          f'{diff:>6} {agreement}')

print()

# Summary
agree_df  = pd.DataFrame(agreements)
strong    = (agree_df['Diff'] <= 3).sum()
moderate  = ((agree_df['Diff'] > 3) &
             (agree_df['Diff'] <= 6)).sum()
disagree  = (agree_df['Diff'] > 6).sum()

print('Agreement Summary:')
print(f'  Strong agreement  (diff ≤ 3) : {strong} features')
print(f'  Moderate agreement(diff ≤ 6) : {moderate} features')
print(f'  Disagreement      (diff > 6) : {disagree} features')
print()

# Top 5 by average rank across both models
agree_df['avg_rank'] = (
    agree_df['LGBM_rank'] + agree_df['XGB_rank']
) / 2
top5 = agree_df.nsmallest(5, 'avg_rank')
print('Top 5 Features by Average Rank (both models):')
print(f'  {"Feature":<25} {"Avg Rank":>10} {"Agreement"}')
print(f'  {"-" * 45}')
for _, row in top5.iterrows():
    print(f'  {row["Feature"]:<25} {row["avg_rank"]:>10.1f} '
          f'{row["Agreement"]}')

print()
print('=' * 65)
print('Feature importance analysis complete')
print('Cross-model top features are strongest signals')
print('for the ensemble in notebook 07')

LightGBM Feature Importance — Split Ranking
(how often each feature is used to split)
  Rank  Feature                     Split % Bar
  -------------------------------------------------------
  1     market_return_1d             11.61%  ███████████
  2     atr_14                        8.46%  ████████
  3     bb_width                      7.67%  ███████
  4     macd_hist                     7.39%  ███████
  5     month                         7.19%  ███████
  6     price_vs_ma50                 6.52%  ██████
  7     return_5d                     6.48%  ██████
  8     return_20d                    6.09%  ██████
  9     volume_trend                  5.96%  █████
  10    return_10d                    5.83%  █████
  11    volume_ratio                  5.74%  █████
  12    rsi_14                        5.70%  █████
  13    price_vs_ma20                 4.58%  ████
  14    return_1d                     3.85%  ███
  15    price_volume_signal           3.61%  ███
  16    day_of_week           

------------

### Cell 11 — Save Model and OOF Predictions

#### What We Save

Four files saved to ../models/ directory.
Same pattern as notebook 04 for consistency.

lgbm_model.pkl
  Final LightGBM model trained on all 24,180 rows.
  Used in notebook 07 to generate validation predictions
  for ensemble evaluation.
  Used in notebook 08 dashboard for live 2025 predictions.

lgbm_oof_probs.npy
  OOF probability array shape (24180, 3).
  Contains P(Sell), P(Hold), P(Buy) for every training row
  predicted by a model that never trained on that row.
  Combined with xgb_oof_probs.npy and lstm_oof_probs.npy
  in notebook 07 to train the Logistic Regression
  meta-model.

lgbm_oof_coverage.npy
  Coverage mask shape (24180,) indicating which rows
  have valid OOF predictions.
  Notebook 07 uses this alongside xgb_oof_coverage.npy
  to find rows predicted by ALL three models.
  Only rows with coverage=1 in all three models
  are used to train the meta-model.

feature_cols.pkl
  Not saved again — already exists from notebook 04.
  We verify it loads correctly and matches our
  current FEATURE_COLS to confirm consistency.

#### Cross-Notebook OOF Compatibility Check

Before saving we run one final check.
We load the XGBoost OOF array from notebook 04
and verify that the LightGBM OOF array is compatible:
  Same shape
  Same rows have valid predictions (coverage matches)
  No row that XGBoost skipped was predicted by LightGBM

This guarantees notebook 07 can safely combine
both OOF arrays to train the meta-model.

In [11]:
# ============================================================
# CELL 11 - Save Model and OOF Predictions
# ============================================================
# Save three new files to ../models/ directory
# Verify feature_cols.pkl matches — do not resave
# Run cross-notebook OOF compatibility check
# Verify every file immediately after saving
# ============================================================

print('Saving Notebook 05 outputs')
print('=' * 55)

# ── File paths ────────────────────────────────────────────
PATH_MODEL    = f'{MODELS_DIR}lgbm_model.pkl'
PATH_OOF_PROB = f'{MODELS_DIR}lgbm_oof_probs.npy'
PATH_OOF_COV  = f'{MODELS_DIR}lgbm_oof_coverage.npy'
PATH_FEATURES = f'{MODELS_DIR}feature_cols.pkl'

# ── Save final model ──────────────────────────────────────
joblib.dump(lgbm_final, PATH_MODEL)
size_model = os.path.getsize(PATH_MODEL) / 1024
print(f'  lgbm_model.pkl saved        : {size_model:>8.1f} KB')

# ── Save OOF probabilities ────────────────────────────────
np.save(PATH_OOF_PROB, oof_probs)
size_oof_prob = os.path.getsize(PATH_OOF_PROB) / 1024
print(f'  lgbm_oof_probs.npy saved    : {size_oof_prob:>8.1f} KB')

# ── Save OOF coverage mask ────────────────────────────────
np.save(PATH_OOF_COV, oof_coverage)
size_oof_cov = os.path.getsize(PATH_OOF_COV) / 1024
print(f'  lgbm_oof_coverage.npy saved : {size_oof_cov:>8.1f} KB')

print()

# ── Verify feature_cols.pkl — do not resave ───────────────
feat_loaded = joblib.load(PATH_FEATURES)
assert feat_loaded == FEATURE_COLS, \
    'FEATURES MISMATCH: feature_cols.pkl differs ' \
    'from current FEATURE_COLS — notebook mismatch'
print(f'  feature_cols.pkl            : verified ✅'
      f'  ({len(feat_loaded)} features — matches notebook 04)')

print()

# ── Verify saved files ────────────────────────────────────
print('Verifying saved files...')
print('-' * 55)

# Verify model
lgbm_loaded  = joblib.load(PATH_MODEL)
sample_rows  = df_val[FEATURE_COLS].values[:10]
orig_preds   = lgbm_final.predict(sample_rows)
loaded_preds = lgbm_loaded.predict(sample_rows)
assert np.array_equal(orig_preds, loaded_preds), \
    'MODEL VERIFY FAILED: loaded model predictions differ'
print(f'  lgbm_model.pkl              : verified ✅'
      f'  (predictions identical on 10 sample rows)')

# Verify OOF probabilities
oof_loaded = np.load(PATH_OOF_PROB)
assert oof_loaded.shape == oof_probs.shape, \
    f'OOF VERIFY FAILED: shape mismatch'
assert np.allclose(oof_loaded, oof_probs, equal_nan=True), \
    'OOF VERIFY FAILED: values differ after reload'
print(f'  lgbm_oof_probs.npy          : verified ✅'
      f'  shape {oof_loaded.shape}')

# Verify OOF coverage
cov_loaded = np.load(PATH_OOF_COV)
assert np.array_equal(cov_loaded, oof_coverage), \
    'COVERAGE VERIFY FAILED: values differ after reload'
print(f'  lgbm_oof_coverage.npy       : verified ✅'
      f'  shape {cov_loaded.shape}')

print()

# ── Cross-notebook OOF compatibility check ────────────────
print('Cross-Notebook OOF Compatibility Check')
print('-' * 55)

xgb_oof_probs    = np.load(f'{MODELS_DIR}xgb_oof_probs.npy')
xgb_oof_coverage = np.load(f'{MODELS_DIR}xgb_oof_coverage.npy')

# Shape must match
assert xgb_oof_probs.shape == oof_probs.shape, \
    f'SHAPE MISMATCH: XGB {xgb_oof_probs.shape} ' \
    f'vs LGBM {oof_probs.shape}'
print(f'  Shape match   : XGB {xgb_oof_probs.shape} '
      f'== LGBM {oof_probs.shape}  ✅')

# Coverage must match — same rows predicted by both
coverage_match = np.array_equal(
    xgb_oof_coverage, oof_coverage
)
assert coverage_match, \
    'COVERAGE MISMATCH: XGB and LGBM predicted ' \
    'different rows — fold structure changed between notebooks'
print(f'  Coverage match: identical rows predicted  ✅')

# Count rows available for meta-model training
# Must have coverage=1 in BOTH models
both_predicted = (
    (xgb_oof_coverage == 1) & (oof_coverage == 1)
).sum()
print(f'  Rows for meta-model training: {both_predicted:,}')
print(f'  (rows predicted by both XGBoost and LightGBM)')

print()

# ── Final summary ──────────────────────────────────────────
print('=' * 55)
print('NOTEBOOK 05 COMPLETE')
print('=' * 55)
print()
print('Files saved to ../models/:')
print(f'  lgbm_model.pkl              '
      f'← load in notebooks 07 and 08')
print(f'  lgbm_oof_probs.npy          '
      f'← load in notebook 07 for stacking')
print(f'  lgbm_oof_coverage.npy       '
      f'← load in notebook 07 for masking')
print()
print('LightGBM Performance Summary:')
print(f'  OOF Mean F1 (macro)   : {np.mean(fold_scores):.4f}')
print(f'  Val F1 (macro)        : {f1_mac:.4f}')
print(f'  Val Balanced Accuracy : {bal_acc:.4f}')
print(f'  Val Overall Accuracy  : {f1_wtd:.4f}')
print()
print('Model Comparison So Far:')
print(f'  {"Model":<12} {"OOF F1":>8} {"Val F1":>8}')
print(f'  {"-" * 30}')
print(f'  {"XGBoost":<12} {"0.3677":>8} {"0.3739":>8}')
print(f'  {"LightGBM":<12} {np.mean(fold_scores):>8.4f} '
      f'{f1_mac:>8.4f}')
print(f'  {"LSTM":<12} {"TBD":>8} {"TBD":>8}')
print(f'  {"Ensemble":<12} {"TBD":>8} {"TBD":>8}')
print()
print('Next notebook : 06_lstm.ipynb')
print('  Load        : ../data/processed/all_stocks_features.csv')
print('  Load        : ../models/feature_cols.pkl')
print('  Save        : ../models/lstm_model.h5')
print('  Save        : ../models/lstm_scaler.pkl')
print('  Save        : ../models/lstm_oof_probs.npy')
print('  Save        : ../models/lstm_oof_coverage.npy')
print('=' * 55)

Saving Notebook 05 outputs
  lgbm_model.pkl saved        :   5080.0 KB
  lgbm_oof_probs.npy saved    :    566.8 KB
  lgbm_oof_coverage.npy saved :    189.0 KB

  feature_cols.pkl            : verified ✅  (17 features — matches notebook 04)

Verifying saved files...
-------------------------------------------------------
  lgbm_model.pkl              : verified ✅  (predictions identical on 10 sample rows)
  lgbm_oof_probs.npy          : verified ✅  shape (24180, 3)
  lgbm_oof_coverage.npy       : verified ✅  shape (24180,)

Cross-Notebook OOF Compatibility Check
-------------------------------------------------------
  Shape match   : XGB (24180, 3) == LGBM (24180, 3)  ✅
  Coverage match: identical rows predicted  ✅
  Rows for meta-model training: 20,100
  (rows predicted by both XGBoost and LightGBM)

NOTEBOOK 05 COMPLETE

Files saved to ../models/:
  lgbm_model.pkl              ← load in notebooks 07 and 08
  lgbm_oof_probs.npy          ← load in notebook 07 for stacking
  lgbm_oof_co